## Imports

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import timedelta, datetime
from unidecode import unidecode
import io
import openpyxl
import statistics

from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.patheffects as pe
from matplotlib.patches import Patch

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Image, Paragraph, Spacer, PageBreak
from reportlab.lib import pagesizes
from reportlab.lib.styles import ParagraphStyle
import reportlab.lib.enums as align
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont



## Romania Weather Metrics Comparison by County (Yesterday vs Today vs Tomorrow)

### Processing data

In [ ]:


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)



class WeatherMetricData:
    def __init__(self, day_dataframes, night_dataframes, metric, palettes, legend_label, map_title, value_unit):
        self.dataframes = [day_dataframes, night_dataframes]
        self.metric = metric
        self.legend_label = legend_label
        self.map_title = map_title
        self.value_unit = value_unit

        self.dates = [
            f"{(current_date - timedelta(1)).strftime(r"%d/%m/%Y")} (Yesterday / Observed)",
            f"{datetime.now().date().strftime(r"%d/%m/%Y")} (Today  / Current)",
            f"{(current_date + timedelta(1)).strftime(r"%d/%m/%Y")} (Tomorrow  / Forecast)"
        ]

        self.colormaps = [LinearSegmentedColormap.from_list("day_colors", palettes["day"]), LinearSegmentedColormap.from_list("day_colors", palettes["night"])]
        self.data_period = ["day", "night"]




weather_df = pd.read_excel("../data/processed/WeatherData_processed.xlsx")
current_date = datetime.now().date()
weather_metrics_data = []

metric_definitions = {
    "temp": {
        "agg": "mean",
        "color_palettes": {
            "day": ["#f17e7e", "#7d0b0b"],
            "night": ["#1a2140", "#556e89"]
        },
        "legend_label": "Temperature (°C)",
        "map_title": "Average temperature",
        "value_unit": "°"
    },
    "cloud": {
        "agg": "mean",
        "color_palettes": {
            "day": ["#b7d4f0", "#031f38"],
            "night": ["#b7d4f0", "#031f38"]
        },
        "legend_label": "Cloud cover (%)",
        "map_title": "Average cloud cover",
        "value_unit": "%"
    },
    "wind_kph": {
        "agg": "mean",
        "color_palettes": {
            "day": ["#cbe7e7", "#71bdfc", "#3ed1a5", "#13e213", "#e2de13", "#e28f13", "#e74c30", "#e73030"],
            "night": ["#cbe7e7", "#71bdfc", "#3ed1a5", "#13e213", "#e2de13", "#e28f13", "#e74c30", "#e73030"]
        },
        "legend_label": "Wind speed (kph)",
        "map_title": "Average wind speed",
        "value_unit": " kph"
    },
    "precip_mm": {
        "agg": "sum",
        "color_palettes": {
            "day": ["#b7eef0", "#0D535F"],
            "night": ["#b7eef0", "#0D535F"]
        },
        "legend_label": "Total rainfall (mm / 24h)",
        "map_title": "Cumulative precipitation (24h)",
        "value_unit": " mm"
    },
    "snow_cm": {
        "agg": "sum",
        "color_palettes": {
            "day": ["#dee2e2", "#49666B"],
            "night": ["#dee2e2", "#49666B"]
        },
        "legend_label": "Total snowfall (cm / 24h)",
        "map_title": "Cumulative snowfall (24h)",
        "value_unit": " cm"
    }
}



def filter_day_night(df, is_day, target_date): 
    if is_day:
        df = df.loc[(df["is_day"] == 1) & (df["date_time"].dt.date == target_date)]
        return df
    else:
        next_date = target_date + timedelta(1)
        df = df.loc[(df["is_day"] == 0) & (df["date_time"] > pd.to_datetime(f"{target_date} 12:00:00")) & (df["date_time"] < pd.to_datetime(f"{next_date} 12:00:00"))]
        return df


filtered_weather_dataframes = []
analysis_dates  = [current_date - timedelta(1), current_date, current_date + timedelta(1)]


for is_day in [True, False]:
    for date in analysis_dates:
        filtered_weather_dataframes.append(filter_day_night(weather_df, is_day, date))



for metric_name, metric_info in metric_definitions.items():
    processed_dfs = []

    for df in filtered_weather_dataframes:
        df = df[["county", "city", "county_abbrev", metric_name]]
        df = df.groupby(["county", "city", "county_abbrev"]).agg({metric_name: metric_info["agg"]})
        df = df.reset_index()
        df[metric_name] = round(df[metric_name], 1)
        processed_dfs.append(df)

    

    weather_metrics_data.append(
        WeatherMetricData(
            day_dataframes = processed_dfs[0:3],
            night_dataframes = processed_dfs[3:6],
            metric = metric_name,
            palettes =  metric_info["color_palettes"],
            legend_label = metric_info["legend_label"],
            map_title = metric_info["map_title"],
            value_unit = metric_info["value_unit"]
        )
    )
  


### Data visualization

In [ ]:


class ReportFigure:
    def __init__(self, report_fig_buffer, report_fig_size, default_figure_dpi):
        self.report_fig_buffer = report_fig_buffer
        self.report_fig_size = report_fig_size
        self.default_figure_dpi = default_figure_dpi

    def get_figure_width(self):
        return self.report_fig_size[0] * self.default_figure_dpi

    def get_figure_height(self):
        return self.report_fig_size[1] * self.default_figure_dpi


romania_geodata = gpd.read_file(r"../data/geo_data/romania.geojson")
romania_geodata["name"] = romania_geodata["name"].apply(lambda value: unidecode(value))
romania_geodata = romania_geodata.to_crs(3395)

report_figures = []


for weather_data in weather_metrics_data:
    for time_of_day in range(0, 2):
        figure, axes = plt.subplots(1, 3, figsize=(40, 20), dpi = 100)

        figure.subplots_adjust(top=1, bottom=0, left=0, right=1, wspace=0.01, hspace=0.9)
        figure.tight_layout()

        data_frames = weather_data.dataframes[time_of_day]

        yesterday_df = data_frames[0]
        today_df = data_frames[1]
        tomorrow_df = data_frames[2]
        metric = weather_data.metric

        min_value = min(
            yesterday_df[metric].min(),
            today_df[metric].min(),
            tomorrow_df[metric].min()
        )

        max_value = max(
            yesterday_df[metric].max(),
            today_df[metric].max(),
            tomorrow_df[metric].max()
        )

        normalization = Normalize(vmin=min_value, vmax=max_value)
        colormap = weather_data.colormaps[time_of_day]

        scalar_mappable = ScalarMappable(norm=normalization, cmap=colormap)
        colorbar_axis = figure.add_axes([0.32, 0.175, 0.375, 0.02])
        colorbar = figure.colorbar(scalar_mappable, cax=colorbar_axis, orientation="horizontal")
        colorbar.set_label(weather_data.legend_label, labelpad=22, size=18)

        tick_step = max(1, round((max_value - min_value) / 10, 0))
        colorbar.set_ticks(np.arange(min_value, max_value, tick_step))
        colorbar.ax.tick_params(labelsize=16, colors="#222222", width=1.5)


        figure.suptitle(t = f"{weather_data.map_title} - {weather_data.data_period[time_of_day]}", size = 50, ha = "center", weight = 700)

        figure.text(
            s="Data source: WeatherAPI.com | Aggregated by county",
            x=0,
            y=0.05,
            size=18,
            ha="left"
        )

        for index in range(3):
            merged_geodata = romania_geodata.merge(
                right=data_frames[index],
                how="inner",
                left_on="name",
                right_on="county"
            )

            merged_geodata["centroid"] = merged_geodata.centroid
            merged_geodata["centroid_lon"] = merged_geodata.centroid.x
            merged_geodata["centroid_lat"] = merged_geodata.centroid.y

            merged_geodata = merged_geodata[
                [
                    "county",
                    "county_abbrev",
                    metric,
                    "admin_centre_node_lat",
                    "admin_centre_node_lng",
                    "centroid",
                    "centroid_lon",
                    "centroid_lat",
                    "geometry"
                ]
            ]

            axis = merged_geodata.plot(
                column=metric,
                cmap=colormap,
                norm=normalization,
                edgecolor="#ffffff",
                linewidth=0.5,
                ax=axes[index]
            )

            axis.set_axis_off()
            axis.set_title(
                f"{weather_data.dates[index]}",
                weight=700,
                size=32,
                y=1.1
            )

            for _, row in merged_geodata.iterrows():
                if row["county"] == "Ilfov":
                    continue

                axis.annotate(
                    text=f"{row['county_abbrev']}\n{row[metric]}{weather_data.value_unit}",
                    xy=(row["centroid_lon"], row["centroid_lat"]),
                    horizontalalignment="center",
                    color="#ffffff",
                    size=10,
                    weight=700,
                    bbox=dict(facecolor="#006e90", alpha=0.75)
                )
            
            
        bbox = figure.get_tightbbox(figure.canvas.get_renderer())
        img_buffer = io.BytesIO()
        figure.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close(figure)  
        img_buffer.seek(0)
        report_figures.append(ReportFigure(img_buffer, [bbox.width, bbox.height], 100))


## Romania current temperature and weather status 

In [ ]:



class MapPlotColors:
    def __init__(self, is_day):
        if is_day:
            day_colors = ["#ffd900", "#a00707"]
            self.cmap = LinearSegmentedColormap.from_list("my_gradient", day_colors)
            self.plot_color = "#baecff"
        else:
            night_colors = ["#172247", "#3f9ad6"]
            self.cmap = LinearSegmentedColormap.from_list("my_gradient", night_colors)
            self.plot_color = "#223b57"



condition_images = {}

wb = openpyxl.load_workbook("../data/processed/WeatherData_processed.xlsx")
ws = wb["weather_condition"]

rows = ws.iter_rows()
next(rows) 

for row in rows:
    condition_images[row[0].value] = row[2].value


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


df_weather = pd.read_excel("../data/processed/WeatherData_processed.xlsx")
romania_geo_df = gpd.read_file("../data/geo_data/romania.geojson")

today_date = datetime.now()


df_current_weather = df_weather.loc[(df_weather["date_time"].dt.date == today_date.date()) & (df_weather['date_time'].dt.hour == today_date.hour)]
df_current_weather = df_current_weather[["county_id", "county", "city", "county_abbrev", "date_time", "is_day", "temp", "condition_id"]]


romania_geo_df = romania_geo_df.to_crs(3395)
romania_geo_df["name"] = romania_geo_df["name"].apply(lambda row: unidecode(row))
romania_geo_df["center"] = romania_geo_df.centroid
romania_geo_df['center_lon'] = romania_geo_df.center.x
romania_geo_df['center_lat'] = romania_geo_df.center.y
romania_geo_df = romania_geo_df[["name", "center", "center_lon", "center_lat", "geometry"]]


df_romania = romania_geo_df.merge(right = df_current_weather, how = "inner", left_on = "name", right_on = "county")
df_romania.loc[df_romania["city"] == "Drobeta-Turnu Severin", "city"] = "Drobeta"
df_romania.loc[df_romania["city"] == "Miercurea Ciuc", "city"] = "M. Ciuc"
df_romania.loc[df_romania["city"] == "Cluj-Napoca", "city"] = "Cluj"


map_plot_colors = MapPlotColors(is_day = 1 in df_romania["is_day"].values)


ax = df_romania.plot(color = map_plot_colors.plot_color, figsize = (13,9), edgecolor = "#9fadb3", linewidth = 1.5)

ax.set_axis_off()
fig = ax.get_figure()
fig.subplots_adjust(top=1, bottom=0, left=0, right=1, wspace=0.01, hspace=0.9)
fig.tight_layout()

fig.text(s = "Data source: WeatherAPI.com | Aggregated by county", x = 0, y = 0, size = 12, ha = "left")
ax.set_title(f"{today_date.strftime(r"%d/%m/%Y %H:00")}", weight = 700, size = 23, y = 1)

main_cities_list = ["Iasi", "Suceava", "Galati", "Constanta", "Bucuresti", "Cluj", "Pitesti", "Craiova", "Brasov", "Buzau", "Timisoara", "Oradea", "Satu Mare", "Drobeta", "M. Ciuc", "Sibiu"]

for idx, row in df_romania.iterrows():
    if row["county"] == "Ilfov" or row["city"] not in main_cities_list: continue

    img = plt.imread(f"../images/weather_condition_icons/{row["condition_id"]}.png")
    im = OffsetImage(img, zoom = 0.8)
    ab = AnnotationBbox(im, (row["center_lon"] - 10000, row["center_lat"] + 22500), frameon = False, box_alignment=(0, 0))
    ax.add_artist(ab)
    
    min_temp = df_romania["temp"].min()
    max_temp = df_romania["temp"].max()
    norm = Normalize(vmin = min_temp, vmax = max_temp)

    ax.annotate(
            text = row['city'], 
            xy = (row["center_lon"], row["center_lat"]),
            horizontalalignment='center',
            color = "#ffffff",
            size = 12,
            weight = 700,
            bbox=dict(facecolor='#006e90', alpha=0.65)
            )
    
    ax.annotate(
            text = f"{row['temp']}°", 
            xy = (row["center_lon"] - 10000, row["center_lat"] + 35000),
            horizontalalignment='right',
            color = map_plot_colors.cmap(norm(row['temp'])),
            size = 15,
            weight = 700,
            path_effects = [pe.withStroke(linewidth = 2, foreground = "#ffffff")]
            )
    

bbox = fig.get_tightbbox(fig.canvas.get_renderer())
img_buffer = io.BytesIO()
fig.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
plt.close(fig)  
img_buffer.seek(0)
romania_current_weather_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100)

## Romania weather per county
 

In [ ]:
class CurrentTemperature:
    def __init__(self, report_figure, temp_list, time_list, feelslike_temp_list):
        self.report_figure = report_figure
        self.min_temp = min(temp_list)
        self.min_temp_hour = time_list[temp_list.index(min(temp_list))]
        self.max_temp = max(temp_list)
        self.max_temp_hour = time_list[temp_list.index(max(temp_list))]
        self.avg_temp = round(sum(temp_list) / len(temp_list), 1)
        self.amplitude_temp = round(max(temp_list) - min(temp_list), 1)
        self.avg_feels_like_delta = round((sum(feelslike_temp_list) / len(feelslike_temp_list)) - (sum(temp_list) / len(temp_list)), 1)
        self.avg_feels_like = round(sum(feelslike_temp_list) / len(feelslike_temp_list), 1)
        self.temp_list = temp_list
        self.time_list = time_list
        self.feelslike_temp_list = feelslike_temp_list

    def get_temperature_threshold_statistics(self):
        final_result = []

        temperature_thresholds = [-10, 0, 30, 35]
        temperature_thresholds_summary = ["Severe freezing temperatures below −10°C were recorded during the interval: ", 
                                      "Temperatures below 0°C occurred during the following intervals: ",
                                      "High temperatures above 30°C were observed during the following intervals: ",
                                      "Extreme heat conditions were recorded, with temperatures exceeding 35°C during the interval: "]

        for threshold_index in range(4):
            threshold = temperature_thresholds[threshold_index]

            start_time = None
            intervals = []


            for index in range(len(self.temp_list)):
                condition = (self.temp_list[index] < threshold if threshold <= 0 else self.temp_list[index] > threshold)

                if condition:
                    if start_time is None:
                        start_time = self.time_list[index]
                else:
                    if start_time is not None:
                        intervals.append(f"{start_time} - {self.time_list[index]}")
                        start_time = None


            if start_time is not None:
                intervals.append(f"{start_time} - {self.time_list[-1]}")


            if len(intervals) != 0:
                final_result.append(temperature_thresholds_summary[threshold_index] + ", ".join(intervals))
                

        return "<br/>".join(final_result)
           
    def get_thermal_amplitude_report(self):
        if self.amplitude_temp < 5:
            return f"A very small daily thermal amplitude ({self.amplitude_temp}) indicates stable temperatures throughout the day."
        elif self.amplitude_temp < 10:
            return f"A small daily thermal amplitude ({self.amplitude_temp}) indicates relatively stable temperatures throughout the day."
        elif self.amplitude_temp < 15:
            return f"A moderate daily thermal amplitude ({self.amplitude_temp}) reflects normal diurnal temperature variation."
        elif self.amplitude_temp < 20:
            return f"A large daily thermal amplitude ({self.amplitude_temp}) suggests significant temperature variation between day and night."
        else:
            return f"A exceptionally high thermal amplitude ({self.amplitude_temp}) was observed, indicating strong daytime heating and rapid nighttime cooling."

    def get_delta_feelslike_temp_report(self):
        delta = self.avg_feels_like_delta
        abs_delta = abs(self.avg_feels_like_delta)

        if abs_delta <= 1:
            return f"""No significant difference between actual and perceived temperature was observed.
                      Perceived temperature closely matched the actual air temperature."""
        elif abs_delta > 1 and abs_delta <= 4:
            if delta < 0:
                return f"""Wind conditions caused the perceived temperature to be up to {abs_delta}°C lower than the actual air temperature.
                        A moderate wind chill effect was observed, reducing perceived temperatures by up to {abs_delta}°C."""
            else:
                return f"""Feels-like temperatures were up to {abs_delta}°C higher than actual values due to humidity.
                        Moderate heat index effects increased the perceived temperature by up to {abs_delta}°C."""
        elif abs_delta > 4 and abs_delta <= 7:
            if delta < 0:
                return f"""Strong wind chill effects significantly lowered the perceived temperature by up to {abs_delta}°C.
                        Perceived temperatures were significantly colder than actual values due to wind exposure."""
            else:
                return f"""Strong heat index effects caused temperatures to feel up to {abs_delta}°C warmer than actual readings.
                        High humidity led to a significant increase in perceived temperature."""
        elif abs_delta > 7:
            if delta < 0:
                return f"""Severe wind chill conditions were recorded, with perceived temperatures up to {abs_delta}°C lower than actual air temperatures.
                        Dangerous wind chill conditions significantly increased cold stress risk."""
            else:
                return f"""Severe heat index conditions were observed, with perceived temperatures exceeding actual values by more than {abs_delta}°C.
                        Extreme thermal discomfort was recorded due to high humidity and heat."""

    def get_thermal_conditions_report(self):
        if self.avg_temp <= -10:
            return "Thermal conditions are classified as extremely cold"
        elif self.avg_temp > -10 and self.avg_temp <= 0:
            return "Thermal conditions are classified as very cold"
        elif self.avg_temp > 0 and self.avg_temp <= 15:
            return "Thermal conditions are classified as cold"
        elif self.avg_temp > 15 and self.avg_temp <= 25:
            return "Thermal conditions are classified as comfortable"
        elif self.avg_temp > 25 and self.avg_temp <= 35:
            return "Thermal conditions are classified as hot"
        elif self.avg_temp > 35:
            return "Thermal conditions are classified as extreme heat"

    def get_today_temperature_summary(self):
        return f"""Minimum temperatures reach {self.min_temp} °C around {self.min_temp_hour}, 
                while maximum values of {self.max_temp} °C are recorded around {self.max_temp_hour}. 
                The average air temperature is {self.avg_temp}°C, while the perceived (feels-like) temperature is {self.avg_feels_like}°C.""" 

class DailyTemperature:

    def __init__(self, report_figure, min_temp_list, avg_temp_list, max_temp_list, dates_list, daily_temp_variation_values):
        self.report_figure = report_figure
        self.min_temp_list = min_temp_list
        self.avg_temp_list = avg_temp_list
        self.max_temp_list = max_temp_list
        self.dates_list = dates_list
        self.daily_temp_variation_values = daily_temp_variation_values

    def generate_temperature_variability_summary(self):
        std_avg_temp = round(statistics.stdev(self.avg_temp_list), 1)

        if std_avg_temp < 2:
            return f"Temperature variability was low (standard deviation: {std_avg_temp}°C), indicating stable thermal conditions throughout the analyzed period."
        elif std_avg_temp < 4:
            return f"The standard deviation of daily temperatures was {std_avg_temp}°C, suggesting moderate variability with noticeable day-to-day changes."
        else:
            return f"High temperature variability was observed during the period (standard deviation: {std_avg_temp}°C), indicating unstable and rapidly changing thermal conditions."

    def generate_temperature_trend_summary(self):
        past_avg = sum(self.avg_temp_list[3:6]) / 3   
        future_avg = sum(self.avg_temp_list[6:9]) / 3 

        trend = round(future_avg - past_avg, 1)

        if trend > 1.0:
            return f"A warming trend is expected, with average temperatures increasing by approximately {trend}°C over the next days."
        elif trend < -1.0:
            return f"A cooling trend is expected, with average temperatures decreasing by approximately {trend}°C over the next days."
        else:
            return "Temperatures are expected to remain relatively stable over the next days, with no significant warming or cooling trend detected."

    def generate_temperature_classification_summary(self):
        temp_class_summary = ["Extreme cold conditions, with minimum temperatures dropping below −10°C, were recorded on: ", "Freezing temperatures below 0°C, indicating persistent frost conditions, were observed on: ",
                              "Cold thermal conditions dominated the period, with daily mean temperatures between 0 and 12°C recorded on: ", "Comfortable thermal conditions, providing optimal temperature conditions, were observed on: ",
                              "High temperature conditions, with maximum temperatures exceeding 30°C, were recorded on: ", "Extreme heat conditions, indicating a localized heatwave event, were recorded on: "]
        temp_class_dates = [[], [], [], [], [], []]
        temp_class_report = []

        for i in range(9):
            if self.min_temp_list[i] < -10:
                temp_class_dates[0].append(self.dates_list[i]) 
            elif self.min_temp_list[i] >= -10 and self.min_temp_list[i] < 0:
                temp_class_dates[1].append(self.dates_list[i])
            elif self.avg_temp_list[i] >= 0 and self.avg_temp_list[i] < 12:
                temp_class_dates[2].append(self.dates_list[i])
            elif self.avg_temp_list[i] >= 12 and self.avg_temp_list[i] < 25:
                temp_class_dates[3].append(self.dates_list[i])
            elif self.max_temp_list[i] >= 25 and self.max_temp_list[i] <= 30:
                temp_class_dates[4].append(self.dates_list[i])
            elif self.max_temp_list[i] > 30:
                temp_class_dates[5].append(self.dates_list[i])

        

        for i in range(6):
            if len(temp_class_dates[i]) > 0:
                temp_class_report.append(temp_class_summary[i] + ", ".join(temp_class_dates[i]))

        
        return "Temperature Extremes Overview:<br/>" + "<br/>".join(f"-{line}" for line in temp_class_report)

    def generate_daily_temperature_report(self):
        lowest_temperature = min(self.min_temp_list)
        lowest_temperature_date = self.dates_list[self.min_temp_list.index(lowest_temperature)]
        highest_temperature = max(self.max_temp_list)
        highest_temperature_date = self.dates_list[self.max_temp_list.index(highest_temperature)]
        avg_temperature = round(sum(self.avg_temp_list) / len(self.avg_temp_list), 1)
        highest_daily_temp_variation_value = max(self.daily_temp_variation_values)
        highest_daily_temp_variation_date = self.dates_list[self.daily_temp_variation_values.index(highest_daily_temp_variation_value)]

        return f"""During the analyzed period, the lowest temperature was {lowest_temperature}°C, 
                            recorded on {lowest_temperature_date},
                            while the highest temperature reached {highest_temperature}°C on {highest_temperature_date}.
                            The average temperature across the period was {avg_temperature}°C.
                            The largest daily temperature variation {highest_daily_temp_variation_value}°C 
                            was recorded on {highest_daily_temp_variation_date}."""

class DailyPrecipitation:
    def __init__(self, report_figure, dates, precip_mm):
        self.report_figure = report_figure
        self.dates = dates
        self.precip_mm = precip_mm

    def generate_precipitation_trend_summary(self):
        past_daily_precipitation = sum(self.precip_mm[3:6])   
        future_daily_precipitation = sum(self.precip_mm[6:9])
        trend_precipitation = round(future_daily_precipitation - past_daily_precipitation, 2)
        trend_precipitation_summary = None


        if trend_precipitation > 0:
            trend_precipitation_summary = f"Compared to the recent historical period, forecast data suggests increased precipitation over the next days."
        else:
            trend_precipitation_summary = f"Forecast precipitation is expected to be lower than the recent observed totals, indicating a drying trend."

        return trend_precipitation_summary

    def generate_precipitation_summary(self):
        if sum(self.precip_mm) == 0:
            return (False, "No precipitation was recorded during the analyzed period.")
        elif sum(1 for n in self.precip_mm if n != 0) > 1:
            highest_daily_precipitation = max(self.precip_mm)
            highest_daily_precipitation_date = self.dates[self.precip_mm.index(highest_daily_precipitation)]

            lowest_daily_precipitation = min([n for n in self.precip_mm if n != 0])
            lowest_daily_precipitation_date = self.dates[self.precip_mm.index(lowest_daily_precipitation)]

            total_daily_precipitation = round(sum(self.precip_mm), 2)

            return (True, f"""The highest daily precipitation was recorded on {highest_daily_precipitation_date}, with a total of {highest_daily_precipitation} mm.
                       The lowest precipitation amount occurred on {lowest_daily_precipitation_date}, with a total of {lowest_daily_precipitation} mm.
                       The total precipitation, both observed and forecasted, is {total_daily_precipitation} mm.""")
        else:
            for i in range(9):
                if self.precip_mm[i] != 0:
                    return (True, f"During the analyzed period, only one day recorded measurable precipitation: {self.dates[i]} - {self.precip_mm[i]}")
        
    def get_precipitation_hazard_level(self):
        danger_days = []
        for i in range(len(self.precip_mm)):
            if self.precip_mm[i] >= 40:
                danger_days.append(f"{self.dates[i]} ({self.precip_mm[i]} mm / 25h)")

        if len(danger_days) > 0:
            return f"Hazardous precipitation events were recorded on the following days: " + ",".join(danger_days) + "."
        else: 
            return "No hazardous precipitation events were recorded during the analyzed period."

class DailySnowfall:
    def __init__(self, report_figure, dates, snowfall_cm):
        self.report_figure = report_figure
        self.dates = dates
        self.snowfall_cm = snowfall_cm

    def generate_snowfall_trend_summary(self):
        past_daily_snowfall = sum(self.snowfall_cm[3:6])   
        future_daily_snowfall = sum(self.snowfall_cm[6:9])
        trend_snowfall = round(future_daily_snowfall - past_daily_snowfall, 2)
        trend_snowfall_summary = None


        if trend_snowfall > 0:
            trend_snowfall_summary = f"Compared to the recent observed period, forecast data indicates increased snowfall accumulation over the coming days."
        else:
            trend_snowfall_summary = f"Forecast snowfall accumulation is expected to be lower than the recent observed totals, indicating a decreasing snowfall trend."

        return trend_snowfall_summary

    def generate_snowfall_summary(self):
        if sum(self.snowfall_cm) == 0:
            return (False, "No snowfall was recorded during the analyzed period.")
        elif sum(1 for n in self.snowfall_cm if n != 0) > 1:
            highest_daily_snowfall = max(self.snowfall_cm)
            highest_daily_snowfall_date = self.dates[self.snowfall_cm.index(highest_daily_snowfall)]

            lowest_daily_snowfall = min([n for n in self.snowfall_cm if n != 0])
            lowest_daily_snowfall_date = self.dates[self.snowfall_cm.index(lowest_daily_snowfall)]

            total_daily_snowfall = round(sum(self.snowfall_cm), 2)

            return (True, f"""The highest daily snowfall was recorded on {highest_daily_snowfall_date}, with an accumulation of {highest_daily_snowfall} cm.
                       The lowest snowfall amount occurred on {lowest_daily_snowfall_date}, with a total of {lowest_daily_snowfall} cm.
                       The total snowfall accumulation, including observed and forecasted data, is {total_daily_snowfall} cm.""")
        else:
            for i in range(9):
                if self.snowfall_cm[i] != 0:
                    return (True, f"Only one day recorded measurable snowfall during the analyzed period: {self.dates[i]} - {self.snowfall_cm[i]} cm")
        
    def get_snowfall_hazard_level(self):
        danger_days = []
        for i in range(len(self.snowfall_cm)):
            if self.snowfall_cm[i] >= 30:
                danger_days.append(f"{self.dates[i]} ({self.snowfall_cm[i]} cm / 25h)")

        if len(danger_days) > 0:
            return f"Hazardous snowfall events were recorded on the following days: " + ",".join(danger_days) + "."
        else: 
            return "Snowfall accumulation remained below hazardous levels throughout the analyzed period."

class CurrentCloudsAndHumidity:
    def __init__(self, report_figure):
        self.report_figure = report_figure

class CurrentWindSpeed:
    def __init__(self, report_figure):
        self.report_figure = report_figure

class DailyWindSpeed:
    def __init__(self, report_figure):
        self.report_figure = report_figure

class CurrentPrecipitation:
    def __init__(self, report_figure):
        self.report_figure = report_figure

class CurrentSnowfall:
    def __init__(self, report_figure):
        self.report_figure = report_figure



In [ ]:


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


class CountyWeatherData:
    def __get_daily_temperature(self, df, county):
        plt.figure(figsize = (20, 20))
        plt.tight_layout()
        plt.subplots_adjust(hspace = 0.3)

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date


        min_temp = round(min(df_county["temp"].to_list()), 1)
        max_temp = round(max(df_county["temp"].to_list()), 1)

        aggs = ["min", "mean", "max"]
        bar_width = 0.3
        bar_pos = [-bar_width , 0, bar_width]

        temp_classification = {}

        max_temp_list_by_date = df_county.groupby(by = "date").agg({"temp": "max"})["temp"].to_list()
        min_temp_list_by_date = df_county.groupby(by = "date").agg({"temp": "min"})["temp"].to_list()
        avg_temp_list_by_date = df_county.groupby(by = "date").agg({"temp": "mean"})["temp"].to_list()
        dates = [d.strftime("%d/%m/%Y") for d in df_county["date"].unique()]

        daily_temp_variation_values = []
        for i in range(len(max_temp_list_by_date)): daily_temp_variation_values.append(round(max_temp_list_by_date[i] - min_temp_list_by_date[i], 1))
        

        for is_day in range(0, 2):
            for plot_index in range(0, 3):
                df_county_by_date = df_county.loc[df_county["is_day"] == is_day] 
                df_county_temp_bydate = df_county_by_date.groupby(by = "date").agg({"temp": aggs[plot_index]}).reset_index()
                df_county_temp_bydate["temp"] = round(df_county_temp_bydate["temp"], 2)

                agg_temperatures_by_date = df_county_temp_bydate["temp"].to_list()

                bar_colors = []

                for temp in agg_temperatures_by_date:
                    if temp >= 35:
                        color = "#ff0000"
                        temp_class = "very hot"
                        temp_index = 0
                    elif temp < 35 and temp >= 28:
                        color = "#cf520a"
                        temp_class = "hot"
                        temp_index = 1
                    elif temp < 28 and temp >= 22:
                        color = "#ff7300"
                        temp_class = "very warm"
                        temp_index = 2
                    elif temp < 22 and temp >= 16:
                        color = "#ffd000"
                        temp_class = "warm"
                        temp_index = 3
                    elif temp < 16 and temp >= 12:
                        color = "#81ad3b"
                        temp_class = "cool"
                        temp_index = 4
                    elif temp < 12 and temp >= 6:
                        color = "#42caa1"
                        temp_class = "cold"
                        temp_index = 5
                    elif temp < 6 and temp >= 0:
                        color = "#00b7ff"
                        temp_class = "very cold"
                        temp_index = 6
                    elif temp < 0 and temp >= -10:
                        color = "#0051ff"
                        temp_class = "frosty"
                        temp_index = 7
                    elif temp < -10:
                        color = "#1900a6"
                        temp_class = "very frosty"
                        temp_index = 8

                    bar_colors.append(color)
                    if temp_index not in temp_classification: temp_classification[temp_index] = {"class": temp_class, "color": color}



                plt.subplot(2, 1, is_day + 1)
                plt.bar(data = df_county_temp_bydate, x = np.arange(len(dates)) + bar_pos[plot_index], width = bar_width, height = agg_temperatures_by_date, color = bar_colors, zorder = 2,  edgecolor = "black", linewidth = 1)

                ticks_min = min_temp - 5 if min_temp <= 0 else 0
                ticks_max = max_temp + 10
                ticks_step = max(1, round((max_temp - min_temp) / 10, 0))
                ticks_positions_array = np.arange(ticks_min, ticks_max, ticks_step)
                plt.yticks(ticks = ticks_positions_array, fontsize = 8)

                plt.xticks(ticks = range(0, len(dates)), labels = dates, fontsize = 10, style = "italic")
                plt.tick_params(axis='x', pad=20)
                plt.xlabel("Date", labelpad = 12)
                plt.ylabel("Temperature", labelpad = 12)


                ax = plt.gca()
                sec = ax.secondary_xaxis(location=0)
                sec.set_xticks(np.arange(len(dates)) + bar_pos[plot_index], labels = [aggs[plot_index]] * len(dates))



                for index in range(0, len(agg_temperatures_by_date)):
                    plt.text(
                            x = index + bar_pos[plot_index],
                            y = agg_temperatures_by_date[index] + 1 if agg_temperatures_by_date[index] >= 0 else agg_temperatures_by_date[index] - 1,
                            s = f"{agg_temperatures_by_date[index]} C°",
                            ha = "center",
                            fontsize = 8,
                            font = "arial",
                            weight = "bold",
                            zorder = 2)


            temp_classification_keys = list(temp_classification.keys())
            temp_classification_keys.sort()
            temp_classification_sorted = {i: temp_classification[i] for i in temp_classification_keys}

            legend_elements = []
            for key, value in temp_classification_sorted.items():
                legend_elements.append(Patch(facecolor = value["color"], edgecolor = '#000000', label = value["class"]))

            plt.legend(handles = legend_elements, bbox_to_anchor=(1, 0.5), loc = "center left", fontsize = 8)
            plt.grid(zorder = 0, axis = "y")
            day_time = "day" if is_day == 1 else "night"
            plt.title(f"{county} [{day_time}] min, average and max temperature (C°) | {min(dates)} - {max(dates)}", fontsize = 10, pad = 12)


        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)

        return DailyTemperature(
            report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100),
            min_temp_list = min_temp_list_by_date,
            max_temp_list = max_temp_list_by_date,
            avg_temp_list = avg_temp_list_by_date,
            dates_list = dates,
            daily_temp_variation_values = daily_temp_variation_values
            )

    def __get_current_temperature(self, df, county):
        plt.figure(figsize = (20, 7))
        plt.tight_layout()

        today_date = datetime.now().date()

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date
        df_county["time"] = df_county["date_time"].dt.time
        df_county_today_weather = df_county.loc[df_county["date"] == today_date]

        time_list = [time.strftime("%H:%M") for time in df_county_today_weather["time"]]
        temp_list = df_county_today_weather["temp"].to_list()
        feelslike_temp_list = df_county_today_weather["feelslike_c"].to_list()
        
        min_temp = round(min(min(temp_list), min(feelslike_temp_list)), 0)
        max_temp = round(max(max(temp_list), max(feelslike_temp_list)), 0)

        plt.plot(range(0, len(time_list)), temp_list, zorder = 2, marker = 'o', markersize = 5, color = "#559EE2", label = "Temperature")
        plt.plot(range(0, len(feelslike_temp_list)), feelslike_temp_list, zorder = 2, marker = 'o', markersize = 5, color = "#64C9E2", label = "Feels like temperature")
        plt.xticks(ticks = range(0, len(time_list)), labels = time_list, rotation = 45)
        plt.yticks(
                ticks = np.arange(
                    min_temp - 2, 
                    max_temp + 2, 
                    max(1, round((max_temp - min_temp) / 10, 0))),
                fontsize = 8)
        

        plt.ylabel("Temperature (C°)")
        plt.xlabel("Time (hour)")
        plt.title(f"{county} temperature (C°) | {today_date}")
        plt.grid(zorder = 0, axis = "y")
        plt.legend()
        
        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)

        return CurrentTemperature(
            report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100),
            temp_list = temp_list,
            time_list = time_list,
            feelslike_temp_list = feelslike_temp_list
        )

    def __get_current_clouds_and_humidity(self, df, county):
        plt.figure(figsize = (20, 7))
        plt.tight_layout()

        today_date = datetime.now().date()

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date
        df_county["time"] = df_county["date_time"].dt.time
        df_county_today_weather = df_county.loc[df_county["date"] == today_date]

        time_list = [time.strftime("%H:%M") for time in df_county_today_weather["time"]]
        cloud_list = df_county_today_weather["cloud"].to_list()
        humidity_list = df_county_today_weather["humidity"].to_list()
        
        min_value = round(min(min(cloud_list), min(humidity_list)), 0)
        max_value = round(max(max(cloud_list), max(humidity_list)), 0)

        plt.plot(range(0, len(time_list)), cloud_list, zorder = 2, marker = 'o', markersize = 5, color = "#559EE2", label = "Cloud cover as percentage")
        plt.plot(range(0, len(humidity_list)), humidity_list, zorder = 2, marker = '+', markersize = 5, color = "#64C9E2", label = "Humidity as percentage")
        plt.xticks(ticks = range(0, len(time_list)), labels = time_list, rotation = 45)
        plt.yticks(
                ticks = range(0, 105, 5),
                fontsize = 8)
        

        plt.ylabel("Percentage %")
        plt.xlabel("Time (hour)")
        plt.title(f"{county} cloud cover and humidity | {today_date}")
        plt.grid(zorder = 0, axis = "y")
        plt.legend()

        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)

        return CurrentCloudsAndHumidity(report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100))

    def __get_current_windspeed(self, df, county):
        plt.figure(figsize = (20, 7))
        plt.tight_layout()

        today_date = datetime.now().date()

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date
        df_county["time"] = df_county["date_time"].dt.time
        df_county_today_weather = df_county.loc[df_county["date"] == today_date]

        time_list = [time.strftime("%H:%M") for time in df_county_today_weather["time"]]
        wind_list = df_county_today_weather["wind_kph"].to_list()
        wind_gust_list = df_county_today_weather["wind_gust_kph"].to_list()
        
        min_wind_speed = round(min(min(wind_list), min(wind_gust_list)), 0)
        max_wind_speed = round(max(max(wind_list), max(wind_gust_list)), 0)

        plt.plot(range(0, len(time_list)), wind_list, zorder = 2, marker = 'o', markersize = 5, color = "#559EE2", label = "Wind speed")
        plt.plot(range(0, len(time_list)), wind_gust_list, zorder = 2, marker = 'o', markersize = 5, color = "#64C9E2", label = "Wind gust speed")
        plt.xticks(ticks = range(0, len(time_list)), labels = time_list, rotation = 45)
        plt.yticks(
                ticks = np.arange(
                    min_wind_speed - 2, 
                    max_wind_speed + 2, 
                    max(1, round((max_wind_speed - min_wind_speed) / 10, 0))),
                fontsize = 8)
        

        plt.ylabel("Wind speed (kph)")
        plt.xlabel("Time (hour)")
        plt.title(f"{county} wind speed (kph) | {today_date}")
        plt.grid(zorder = 0, axis = "y")
        plt.legend()

        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)
        return CurrentWindSpeed(report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100))

    def __get_daily_windspeed(self, df, county):
        plt.figure(figsize = (20, 17))
        plt.tight_layout()
        plt.subplots_adjust(hspace = 0.3)

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date

        min_wind_speed = round(min(df_county["wind_kph"].to_list()), 0)
        max_wind_speed = round(max(df_county["wind_gust_kph"].to_list()), 0)

        aggs = [{"wind_kph": "mean"}, {"wind_gust_kph": "max"}]
        wind_types = ["wind_kph", "wind_gust_kph"]
        bar_width = 0.3
        bar_pos = [-bar_width , 0]

        wind_classification = {}

        for is_day in range(0, 2):
            for plot_index in range(0, 2):
                wind_type = wind_types[plot_index]
                aggregate_type = aggs[plot_index][wind_type]

                df_county_by_date = df_county.loc[df_county["is_day"] == is_day]

                df_windspeed_by_date = df_county_by_date.groupby(by = "date").agg(aggs[plot_index]).reset_index()
                df_windspeed_by_date[wind_type] = round(df_windspeed_by_date[wind_type], 0)
                wind_speed_list = df_windspeed_by_date[wind_type].to_list()
            
                dates = df_windspeed_by_date["date"].to_list()

                bar_colors = []
        

                for wind_speed in wind_speed_list:
                    if wind_speed >= 117:
                        color = "#ff0000"
                        wind_class = "hurricane"
                        wind_index = 0
                    elif wind_speed <= 116 and wind_speed >= 102:
                        color = "#ff3c00"
                        wind_class = "violent storm"
                        wind_index = 1
                    elif wind_speed <= 101 and wind_speed >= 88:
                        color = "#ee6f07"
                        wind_class = "whole gale"
                        wind_index = 2
                    elif wind_speed <= 87 and wind_speed >= 75:
                        color = "#d88509"
                        wind_class = "strong gale"
                        wind_index = 3
                    elif wind_speed <= 74 and wind_speed >= 62:
                        color = "#ffbb00"
                        wind_class = "fresh gale"
                        wind_index = 4
                    elif wind_speed <= 61 and wind_speed >= 51:
                        color = "#ffe600"
                        wind_class = "moderate gale"
                        wind_index = 5
                    elif wind_speed <= 50 and wind_speed >= 40:
                        color = "#b1ce11"
                        wind_class = "strong breeze"
                        wind_index = 6
                    elif wind_speed <= 39 and wind_speed >= 30:
                        color = "#a6fd03"
                        wind_class = "fresh breeze"
                        wind_index = 7
                    elif wind_speed <= 29 and wind_speed >= 20:
                        color = "#15d344"
                        wind_class = "moderate breeze"
                        wind_index = 8
                    elif wind_speed <= 19 and wind_speed >= 12:
                        color = "#0cc770"
                        wind_class = "gentle breeze"
                        wind_index = 9
                    elif wind_speed <= 11 and wind_speed >= 6:
                        color = "#03f7c2"
                        wind_class = "light breeze"
                        wind_index = 10
                    elif wind_speed <= 5 and wind_speed >= 2:
                        color = "#63d5dd"
                        wind_class = "light air"
                        wind_index = 11
                    elif wind_speed < 2:
                        color = "#b1c1df"
                        wind_class = "calm"
                        wind_index = 12

                    bar_colors.append(color)
                    if wind_index not in wind_classification: wind_classification[wind_index] = {"class": wind_class, "color": color}

                
                plt.subplot(2, 1, is_day + 1)

                plt.bar(data = df_windspeed_by_date, x =  np.arange(len(dates)) + bar_pos[plot_index], width = bar_width, height = wind_speed_list, color = bar_colors, zorder = 2,  edgecolor = "black", linewidth = 1)

                plt.yticks(
                    ticks = np.arange(
                        min_wind_speed - 5 if min_wind_speed <= 0 else 0, 
                        max_wind_speed + 10, 
                        max(1, round((max_wind_speed - min_wind_speed) / 10, 0))),
                    fontsize = 8)

                plt.xticks(ticks = np.arange(0, len(dates)) - bar_width / 2, labels = dates, fontsize = 9, style = "italic")
                plt.tick_params(axis='x', pad=20)
                plt.xlabel("Date", labelpad = 12)
                plt.ylabel("Wind speed (kph)", labelpad = 12)

                ax = plt.gca()
                sec = ax.secondary_xaxis(location=0)
                sec.set_xticks(np.arange(len(dates)) + bar_pos[plot_index], labels = [aggregate_type] * len(dates))


                for index in range(0, len(wind_speed_list)):
                    plt.text(
                            x = index + bar_pos[plot_index],
                            y = wind_speed_list[index] + 1 if wind_speed_list[index] >= 0 else wind_speed_list[index] - 2,
                            s = f"{wind_speed_list[index]} kph",
                            ha = "center",
                            fontsize = 7,
                            font = "arial",
                            weight = "bold",
                            zorder = 2)


            wind_classification_keys = list(wind_classification.keys())
            wind_classification_keys.sort()
            wind_classification_sorted = {i: wind_classification[i] for i in wind_classification_keys}
            
            legend_elements = []
            for key, value in wind_classification_sorted.items():
                legend_elements.append(Patch(facecolor = value["color"], edgecolor = '#000000', label = value["class"]))

            plt.legend(handles = legend_elements, bbox_to_anchor=(1, 0.5), loc = "center left", fontsize = 8)
            plt.grid(zorder = 0, axis = "y")
            day_time = "day" if is_day == 1 else "night"
            plt.title(f"{county} [{day_time}] average and max wind speed (kph) | {min(dates)} - {max(dates)}", fontsize = 10, pad = 12)


        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)

        return DailyWindSpeed(report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100))

    def __get_daily_precipitation(self, df, county):
        plt.figure(figsize = (12, 8))
        plt.tight_layout()
        

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date


        precip_classification = {}

        df_county_precip_bydate = df_county.groupby(by = "date").agg({"precip_mm": "sum"}).reset_index()
        df_county_precip_bydate["precip_mm"] = round(df_county_precip_bydate["precip_mm"], 2)

        precip_mm = df_county_precip_bydate["precip_mm"].to_list()
        dates = df_county_precip_bydate["date"].to_list()

        min_precip = round(min(precip_mm), 2)
        max_precip = round(max(precip_mm), 2)


        bar_colors = []

        for precip in precip_mm:
            if precip >= 100:
                color = "#070067"
                precip_class = "exceptional rainfall"
                precip_index = 0
            elif precip < 100 and precip >= 80:
                color = "#181179"
                precip_class = "extreme rainfall"
                precip_index = 1
            elif precip < 80 and precip >= 60:
                color = "#110895"
                precip_class = "excessive rainfall"
                precip_index = 2
            elif precip < 60 and precip >= 40:
                color = "#2C21CA"
                precip_class = "very heavy rain"
                precip_index = 3
            elif precip < 40 and precip >= 30:
                color = "#4337DF"
                precip_class = "heavy rain"
                precip_index = 4
            elif precip < 30 and precip >= 20:
                color = "#3C64E6"
                precip_class = "notable rainfall"
                precip_index = 5
            elif precip < 20 and precip >= 10:
                color = "#597DF2"
                precip_class = "moderate rain"
                precip_index = 6
            elif precip < 10 and precip >= 5:
                color = "#80AEF3"
                precip_class = "light rainfall"
                precip_index = 7
            elif precip < 5 and precip >= 1:
                color = "#ABCEF4"
                precip_class = "light rain"
                precip_index = 8
            elif precip < 1:
                color = "#C8E5EC"
                precip_class = "very light rain"
                precip_index = 9

            bar_colors.append(color)
            if precip_index not in precip_classification: precip_classification[precip_index] = {"class": precip_class, "color": color}


        plt.bar(data = df_county_precip_bydate, x = np.arange(len(dates)), width = 0.5, height = precip_mm, color = bar_colors, zorder = 2,  edgecolor = "black", linewidth = 1)

        ticks_min = min_precip
        ticks_max = max_precip + 1
        ticks_step = max(0.1, round(max_precip / 10, 2))
        ticks_positions_array = np.arange(ticks_min, ticks_max, ticks_step)
        plt.yticks(ticks = ticks_positions_array, fontsize = 8)

        plt.xticks(ticks = range(0, len(dates)), labels = dates, fontsize = 10, style = "italic")
        plt.tick_params(axis='x', pad=20)
        plt.xlabel("Date", labelpad = 12)
        plt.ylabel("Daily Total Precipitation (mm)", labelpad = 12)


    

        for index in range(0, len(precip_mm)):
            plt.text(
                    x = index,
                    y = precip_mm[index] + 0.1,
                    s = f"{precip_mm[index]} mm",
                    ha = "center",
                    fontsize = 8,
                    font = "arial",
                    weight = "bold",
                    zorder = 2)


        precip_classification_keys = list(precip_classification.keys())
        precip_classification_keys.sort()
        precip_classification_sorted = {i: precip_classification[i] for i in precip_classification_keys}

        legend_elements = []
        for key, value in precip_classification_sorted.items():
            legend_elements.append(Patch(facecolor = value["color"], edgecolor = '#000000', label = value["class"]))

        plt.legend(handles = legend_elements, bbox_to_anchor=(1, 0.5), loc = "center left", fontsize = 8)
        plt.grid(zorder = 0, axis = "y")
        plt.title(f"{county} daily rainfall (mm / 24h) | {min(dates)} - {max(dates)}", fontsize = 10, pad = 12)

        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)

        return DailyPrecipitation(
            report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100),
            precip_mm = precip_mm,
            dates = dates
            )

    def __get_current_precipitation(self, df, county):
        plt.figure(figsize = (20, 7))
        plt.tight_layout()

        today_date = datetime.now().date()

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date
        df_county["time"] = df_county["date_time"].dt.time
        df_county_today_weather = df_county.loc[df_county["date"] == today_date]

        time_list = [time.strftime("%H:%M") for time in df_county_today_weather["time"]]
        precip_list = df_county_today_weather["precip_mm"].to_list()
        
        
        min_precip = round(min(precip_list), 0)
        max_precip = round(max(precip_list), 0)

        plt.plot(range(0, len(time_list)), precip_list, zorder = 2, marker = 'o', markersize = 5, color = "#559EE2", label = "precip_mm")
        plt.xticks(ticks = range(0, len(time_list)), labels = time_list, rotation = 45)

        ticks_min = min_precip
        ticks_max = max_precip + 1
        ticks_step = max(0.1, round(max_precip / 10, 2))
        ticks = np.arange(ticks_min, ticks_max, ticks_step)
        plt.yticks(ticks = ticks, fontsize = 8)
        

        plt.ylabel("Precipitation (mm / hour)")
        plt.xlabel("Time (hour)")
        plt.title(f"{county} hourly rainfall | {today_date}")
        plt.grid(zorder = 0, axis = "y")
        plt.legend()

        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)
        return CurrentPrecipitation(report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100))

    def __get_daily_snowfall(self, df, county):
        plt.figure(figsize = (12, 8))
        plt.tight_layout()
        

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date


        snow_classification = {}

        df_county_snow_bydate = df_county.groupby(by = "date").agg({"snow_cm": "sum"}).reset_index()
        df_county_snow_bydate["snow_cm"] = round(df_county_snow_bydate["snow_cm"], 2)

        snow_cm = df_county_snow_bydate["snow_cm"].to_list()
        dates = df_county_snow_bydate["date"].to_list()

        min_snow = round(min(snow_cm), 2)
        max_snow = round(max(snow_cm), 2)


        bar_colors = []

        for snow in snow_cm:
            if snow >= 70:
                color = "#070067"
                snow_class = "exceptional snowfall"
                snow_index = 0
            elif snow >= 50:
                color = "#181179"
                snow_class = "extreme snowfall"
                snow_index = 1
            elif snow >= 30:
                color = "#110895"
                snow_class = "very heavy snowfall"
                snow_index = 2
            elif snow >= 20:
                color = "#2C21CA"
                snow_class = "heavy snowfall"
                snow_index = 3
            elif snow >= 10:
                color = "#4337DF"
                snow_class = "moderate snowfall"
                snow_index = 4
            elif snow >= 5:
                color = "#3C64E6"
                snow_class = "light snowfall"
                snow_index = 5
            elif snow >= 1:
                color = "#597DF2"
                snow_class = "very light snowfall"
                snow_index = 6
            else:
                color = "#C8E5EC"
                snow_class = "no snow"
                snow_index = 7


            bar_colors.append(color)
            if snow_index not in snow_classification: snow_classification[snow_index] = {"class": snow_class, "color": color}


        plt.bar(data = df_county_snow_bydate, x = np.arange(len(dates)), width = 0.5, height = snow_cm, color = bar_colors, zorder = 2,  edgecolor = "black", linewidth = 1)

        ticks_min = min_snow
        ticks_max = max_snow + 1
        ticks_step = max(0.1, round(max_snow / 10, 2))
        ticks_positions_array = np.arange(ticks_min, ticks_max, ticks_step)
        plt.yticks(ticks = ticks_positions_array, fontsize = 8)

        plt.xticks(ticks = range(0, len(dates)), labels = dates, fontsize = 10, style = "italic")
        plt.tick_params(axis='x', pad=20)
        plt.xlabel("Date", labelpad = 12)
        plt.ylabel("Daily total snowfall (cm)", labelpad = 12)


    

        for index in range(0, len(snow_cm)):
            plt.text(
                    x = index,
                    y = snow_cm[index] + 0.1,
                    s = f"{snow_cm[index]} cm",
                    ha = "center",
                    fontsize = 8,
                    font = "arial",
                    weight = "bold",
                    zorder = 2)


        snow_classification_keys = list(snow_classification.keys())
        snow_classification_keys.sort()
        snow_classification_sorted = {i: snow_classification[i] for i in snow_classification_keys}

        legend_elements = []
        for key, value in snow_classification_sorted.items():
            legend_elements.append(Patch(facecolor = value["color"], edgecolor = '#000000', label = value["class"]))

        plt.legend(handles = legend_elements, bbox_to_anchor=(1, 0.5), loc = "center left", fontsize = 8)
        plt.grid(zorder = 0, axis = "y")
        plt.title(f"{county} daily snowfall (cm / 24h) | {min(dates)} - {max(dates)}", fontsize = 10, pad = 12)

        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)
        
        return DailySnowfall(
            report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100),
            dates = dates,
            snowfall_cm = snow_cm
        )

    def __get_current_snowfall(self, df, county):
        plt.figure(figsize = (20, 7))
        plt.tight_layout()

        today_date = datetime.now().date()

        df_county = df.loc[df["county"] == county].copy()
        df_county["date"] = df_county["date_time"].dt.date
        df_county["time"] = df_county["date_time"].dt.time
        df_county_today_weather = df_county.loc[df_county["date"] == today_date]

        time_list = [time.strftime("%H:%M") for time in df_county_today_weather["time"]]
        snow_list = df_county_today_weather["snow_cm"].to_list()
        
        
        min_snow = round(min(snow_list), 0)
        max_snow = round(max(snow_list), 0)

        plt.plot(range(0, len(time_list)), snow_list, zorder = 2, marker = 'o', markersize = 5, color = "#559EE2", label = "snow_mm")
        plt.xticks(ticks = range(0, len(time_list)), labels = time_list, rotation = 45)

        ticks_min = min_snow
        ticks_max = max_snow + 1
        ticks_step = max(0.1, round(max_snow / 10, 2))
        ticks = np.arange(ticks_min, ticks_max, ticks_step)
        plt.yticks(ticks = ticks, fontsize = 8)
        

        plt.ylabel("Snowfall (cm / hour)")
        plt.xlabel("Time (hour)")
        plt.title(f"{county} hourly snowfall | {today_date}")
        plt.grid(zorder = 0, axis = "y")
        plt.legend()

        fig = plt.gcf()
        bbox = fig.get_tightbbox(fig.canvas.get_renderer())
        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format="png", dpi=100, bbox_inches="tight")
        plt.close()  
        img_buffer.seek(0)
        return CurrentSnowfall(report_figure = ReportFigure(img_buffer, [bbox.width, bbox.height], 100))

    def __init__(self, county_name, data_frame):
        self.county_name = county_name
        self.data_frame = data_frame

        self.current_snowfall = self.__get_current_snowfall(data_frame, county_name)
        self.daily_snowfall = self.__get_daily_snowfall(data_frame, county_name)
        self.current_precipitation = self.__get_current_precipitation(data_frame, county_name)
        self.daily_precipitation = self.__get_daily_precipitation(data_frame, county_name)
        self.daily_temperature = self.__get_daily_temperature(data_frame, county_name)
        self.current_temperature = self.__get_current_temperature(data_frame, county_name)
        self.current_clouds_and_humidity = self.__get_current_clouds_and_humidity(data_frame, county_name)
        self.current_windspeed = self.__get_current_windspeed(data_frame, county_name)
        self.daily_windspeed = self.__get_daily_windspeed(data_frame, county_name)


df_weather = pd.read_excel(r"../data/processed/WeatherData_processed.xlsx")
df_geo_romania = gpd.read_file(r"../data/geo_data/romania.geojson")

df_geo_romania["name"] = df_geo_romania["name"].apply(lambda row: unidecode(row))
df_geo_romania = df_geo_romania[["name", "admin_centre_node_lat", "admin_centre_node_lng", "geometry"]]

df = pd.merge(left = df_weather, right = df_geo_romania, how = "inner", left_on = "county", right_on = "name")
df = df.drop('name', axis = 1)

counties = [
    'Bistrita-Nasaud', 'Maramures', 'Satu Mare', 'Salaj', 'Bihor', 'Cluj', 'Alba',
    'Bucuresti', 'Suceava', 'Mures', 'Sibiu', 'Brasov', 'Covasna', 'Harghita',
    'Neamt', 'Bacau', 'Arad', 'Hunedoara', 'Botosani', 'Iasi', 'Vaslui', 'Vrancea',
    'Galati', 'Gorj', 'Valcea', 'Arges', 'Dambovita', 'Prahova', 'Buzau', 'Braila',
    'Ialomita', 'Timis', 'Caras-Severin', 'Mehedinti', 'Dolj', 'Olt', 'Teleorman',
    'Giurgiu', 'Ilfov', 'Calarasi', 'Constanta', 'Tulcea'
]

counties_weather_data = []

for county in counties: counties_weather_data.append(CountyWeatherData(county_name = county, data_frame = df))
    



## Generate PDF (report)

In [ ]:

def draw_on_page(canvas, doc):
    global page_number
    page_number += 1

    if page_number > 0:
        canvas.saveState()
        canvas.setFont("Inter-Italic", 9)
        canvas.drawCentredString(pagesizes.A4[0] / 2, 15, f"{page_number} / {total_pages}")
        canvas.restoreState()

def add_image(report_figure: ReportFigure, size: float, space: float = 15):
        img_width = (report_figure.get_figure_width()) * size
        img_height = (report_figure.get_figure_height()) * size
        img = Image(report_figure.report_fig_buffer, width = img_width, height = img_height)

        Story.append(img)
        Story.append(Spacer(1, space))
    
def add_paragraph(text_style: ParagraphStyle, text: str, space: float = 5):
    report_text = Paragraph(text = text, style = text_style)

    Story.append(report_text)
    Story.append(Spacer(1, space))


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


doc = SimpleDocTemplate(filename = r"../WeatherReport.pdf", pagesize = pagesizes.A4, rightMargin = 25, leftMargin = 25, topMargin = 15, bottomMargin = 15)
Story = []

pdfmetrics.registerFont(TTFont(name = "Inter-Regular", filename = r"../fonts/Inter/static/Inter_18pt-Regular.ttf"))
pdfmetrics.registerFont(TTFont(name = "Inter-SemiBold", filename = r"../fonts/Inter/static/Inter_18pt-SemiBold.ttf"))
pdfmetrics.registerFont(TTFont(name = "Inter-Italic", filename = r"../fonts/Inter/static/Inter_18pt-Italic.ttf"))
pdfmetrics.registerFont(TTFont(name = "Inter-SemiBoldItalic", filename = r"../fonts/Inter/static/Inter_18pt-SemiBoldItalic.ttf"))
print(f"Registered fonts: {pdfmetrics.getRegisteredFontNames()}")


Story.append(Spacer(1, 50))

weather_bulletin_title_style = ParagraphStyle(name = "wb_title_style", fontName = "Inter-SemiBold", fontSize = 25, alignment = align.TA_CENTER, leading = 35)
weather_bulletin_title = Paragraph(text = "Romania Weather Bulletin", style = weather_bulletin_title_style)
Story.append(weather_bulletin_title)

weather_bulletin_period_style = ParagraphStyle(name = "wb_period_style", fontName = "Inter-Regular", fontSize = 12, alignment = align.TA_CENTER, leading = 75)
weather_bulletin_period = Paragraph(text = "Historical data (last 6 days) - Current conditions - Forecast (next 2 days)", style = weather_bulletin_period_style)
Story.append(weather_bulletin_period)

romania_map_image_width = 1280
romania_map_image_height = 922
romania_map_image = Image(
    filename = r"../images/Regiuni_de_dezvoltare.svg.png", 
    width = romania_map_image_width / 2.5, 
    height = romania_map_image_height / 2.5)
Story.append(romania_map_image)

Story.append(Spacer(1, 100))

data_source_style = ParagraphStyle(name = "data_source_style", fontName = "Inter-Italic", fontSize = 10, alignment = align.TA_LEFT, leading = 15)

today_date = datetime.today().strftime("%d-%B-%Y")
generated_date = Paragraph(text = f"Generated on: {today_date}", style = data_source_style)
Story.append(generated_date)

data_source = Paragraph(text = f"Data source: weatherapi.com / osm-boundaries.com", style = data_source_style)
Story.append(data_source)

cover_map_source = Paragraph(text = f"Cover map of Romania: Source: wikipedia.org", style = data_source_style)
Story.append(cover_map_source)

Story.append(PageBreak())



page_number = -1
total_pages = 1 + int(len(report_figures) / 2) + int(len(counties_weather_data) * 6)


titles = ["Romania Temperature Overview", "Romania Cloud Coverage Overview", "Romania Wind Speed Overview", "Romania Precipitation Overview", "Romania Snow Depth Overview"]
title_index = 0
page_title_style = ParagraphStyle(name = "page_title_style", fontName = "Inter-SemiBoldItalic", fontSize = 15, alignment = align.TA_CENTER, leading = 50)
report_text_style = ParagraphStyle(name = "report_text_style", fontName = "Inter-Regular", fontSize = 8, alignment = align.TA_LEFT, leading = 12)


page_title = Paragraph(text = "Romania current weather", style = page_title_style)
Story.append(page_title)
Story.append(Spacer(1, 75))

img_width = (romania_current_weather_figure.get_figure_width()) / 2.2
img_height = (romania_current_weather_figure.get_figure_height()) / 2.2

img = Image(romania_current_weather_figure.report_fig_buffer, width = img_width, height = img_height)
Story.append(img)
Story.append(PageBreak())


for index in range(0, len(report_figures), 2):
    page_title = Paragraph(text = titles[title_index], style = page_title_style)
    Story.append(page_title)

    img_width = (report_figures[index].get_figure_width()) / 6.8
    img_height = (report_figures[index].get_figure_height()) / 6.8

    img1 = Image(report_figures[index].report_fig_buffer, width = img_width, height = img_height)
    img2 = Image(report_figures[index + 1].report_fig_buffer, width = img_width, height = img_height)    
    Story.append(img1)
    Story.append(Spacer(1, 75))
    Story.append(img2)
    Story.append(PageBreak())

    title_index += 1

for county_weather_data in counties_weather_data:
    current_temperature = county_weather_data.current_temperature
    daily_temperature = county_weather_data.daily_temperature
    current_precipitation = county_weather_data.current_precipitation
    daily_precipitation = county_weather_data.daily_precipitation
    current_snowfall = county_weather_data.current_snowfall
    daily_snowfall = county_weather_data.daily_snowfall
    current_windspeed = county_weather_data.current_windspeed
    daily_windspeed = county_weather_data.daily_windspeed
    current_clouds_and_humidity = county_weather_data.current_clouds_and_humidity

    #Page title
    page_title = Paragraph(text = county_weather_data.county_name, style = page_title_style)
    Story.append(page_title)


    #current_temperature
    add_image(report_figure = current_temperature.report_figure, size = 0.357, space = 25)
    add_paragraph(text_style = report_text_style, text = current_temperature.get_today_temperature_summary(), space = 5)
    add_paragraph(text_style = report_text_style, text = current_temperature.get_thermal_conditions_report(), space = 15)
    add_paragraph(text_style = report_text_style, text = current_temperature.get_thermal_amplitude_report(), space = 0)
    add_paragraph(text_style = report_text_style, text = current_temperature.get_temperature_threshold_statistics(), space = 15)
    add_paragraph(text_style = report_text_style, text = current_temperature.get_delta_feelslike_temp_report(), space = 5)


    Story.append(PageBreak())


    #daily_temperature    
    add_image(report_figure = daily_temperature.report_figure, size = 0.333, space = 25)
    add_paragraph(text_style = report_text_style, text = daily_temperature.generate_daily_temperature_report(), space = 5)
    add_paragraph(text_style = report_text_style, text = daily_temperature.generate_temperature_variability_summary(), space = 5)
    add_paragraph(text_style = report_text_style, text = daily_temperature.generate_temperature_trend_summary(), space = 5)    
    add_paragraph(text_style = report_text_style, text = daily_temperature.generate_temperature_classification_summary(), space = 5)


    Story.append(PageBreak())


    #current_precipitation    
    add_image(report_figure = current_precipitation.report_figure, size = 0.357, space = 15)
    #daily_precipitation
    add_image(report_figure = daily_precipitation.report_figure, size = 0.4, space = 5)

    is_raining, precipitation_summary = daily_precipitation.generate_precipitation_summary()
    add_paragraph(text_style = report_text_style, text = precipitation_summary, space = 5)

    if (is_raining):
        add_paragraph(text_style = report_text_style, text = daily_precipitation.generate_precipitation_trend_summary(), space = 5)
        add_paragraph(text_style = report_text_style, text = daily_precipitation.get_precipitation_hazard_level(), space = 5)


    Story.append(PageBreak())
    

    #current_windspeed
    add_image(report_figure = current_windspeed.report_figure, size = 0.3077, space = 15)
    #daily_windspeed
    add_image(report_figure = daily_windspeed.report_figure, size = 0.3077, space = 5)


    Story.append(PageBreak())
    

    #current_snowfall    
    add_image(report_figure = current_snowfall.report_figure, size = 0.357, space = 15)
    #daily_snowfall
    add_image(report_figure = daily_snowfall.report_figure, size = 0.4, space = 5)

    is_snowing, snowfall_summary = daily_snowfall.generate_snowfall_summary()
    add_paragraph(text_style = report_text_style, text = snowfall_summary, space = 5)

    if (is_snowing):
        add_paragraph(text_style = report_text_style, text = daily_snowfall.generate_snowfall_trend_summary(), space = 5)
        add_paragraph(text_style = report_text_style, text = daily_snowfall.get_snowfall_hazard_level(), space = 5)

    
    Story.append(PageBreak())

    
    #current_clouds_and_humidity
    add_image(report_figure = current_clouds_and_humidity.report_figure, size = 0.3077, space = 0)


    Story.append(PageBreak())


doc.build(Story, onFirstPage=draw_on_page, onLaterPages=draw_on_page)